# 08. Monte Carlo Kockázatelemzés

**Cél**: A kőbányai ingatlanbefektetések sztochasztikus kockázatértékelése Monte Carlo szimulációval (10 000 iteráció).

**Három összehasonlító szcenárió**:
1. **Konzervatív alapszcenárió (Tisztán bérleti fókusz)**: Minimális reálérték-növekedéssel (20 év alatt 1,2× terminális szorzó). Rávilágít, hogy a bérleti díj önmagában 5%-os diszkontráta és költségek mellett mérsékelt eséllyel nyújt pozitív NPV-t.
2. **Városmegújítási Total Return szcenárió (Mázsa tér TOD hatás)**: Évi 3–3,5% reál felértékelődéssel (20 év alatt 1,8× terminális szorzó), amely mellett a tőkenövekmény aktiválja a befektetés valódi megtérülését ($P(\text{NPV}>0) > 80\%$).
3. **Stagflációs / Recessziós Stressz-teszt**: Magasabb költségek (20%), alacsonyabb kihasználtság (80%), csökkenő reálár (-10%), magasabb diszkontráta (7%).

**Kockázati metrikák és Sztochasztikus Architektúra**:
- **Korrelált Változók**: A vételár és a havi bérleti díj bizonytalansági sokkja többváltozós normális eloszlással (`multivariate_normal`, empirikus $r = 0,65$ kovarianciával) generált, elkerülve a függetlenségi feltevés életszerűtlenségét.
- **Value at Risk (VaR 95%)**: A maximális várható veszteség 95%-os megbízhatósági szinten.
- **Conditional VaR (CVaR 95% / Expected Shortfall)**: A VaR küszöböt meghaladó legrosszabb 5%-os kimenetelek átlagos vesztesége.
- **Nyereségesség valószínűsége**: $P(\text{NPV} > 0)$.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado']
kiado = df[df['listing_type'] == 'kiado']

base_price = elado['nm_ar_huf'].median() * 50  # 50 m² lakás vételár
base_rent = kiado['price_huf'].median() if not kiado.empty else 250000
np.random.seed(42)
print("Monte Carlo szimulációs motor kész.")

### 1. Kettős Szcenárió Eredmények és Kockázati KPI Kártyák
10 000 véletlenszerű piaci pálya szimulációja: a tisztán bérleti pálya, a Mázsa téri városmegújítási tőkenövekmény és a stagflációs stressz-teszt összehasonlítása.

In [ ]:
N_ITERS = 10000
# Korrelált sztochasztikus sokkok (Ár és Bérlet közötti empirikus r = 0.65 korreláció)
mean_vec = [base_price, base_rent]
std_price = base_price * 0.12
std_rent = base_rent * 0.10
corr = 0.65
cov_matrix = [
    [std_price**2, corr * std_price * std_rent],
    [corr * std_price * std_rent, std_rent**2]
]
corr_shocks = np.random.multivariate_normal(mean_vec, cov_matrix, size=N_ITERS)
price_shocks = corr_shocks[:, 0]
rent_shocks = corr_shocks[:, 1]
occ_shocks = np.clip(np.random.normal(loc=0.92, scale=0.06, size=N_ITERS), 0.70, 1.00)
disc_rate = 0.05
op_cost_ratio = 0.15

# 20 éves NPV számítás
eves_netto_cf = rent_shocks * 12 * occ_shocks * (1.0 - op_cost_ratio)
annuity_factor = (1.0 - (1.0 + disc_rate) ** -20) / disc_rate

# 1. Konzervatív pálya (1.2x terminális szorzó)
term_base = price_shocks * 1.20 / ((1.0 + disc_rate) ** 20)
npv_base = (eves_netto_cf * annuity_factor + term_base) - price_shocks

# 2. Városmegújítási Total Return pálya (1.8x terminális szorzó TOD felértékelődéssel)
term_ren = price_shocks * 1.80 / ((1.0 + disc_rate) ** 20)
npv_renewal = (eves_netto_cf * annuity_factor + term_ren) - price_shocks

# 3. Stagflációs / Recessziós Stressz-teszt
stress_occ = np.clip(np.random.normal(loc=0.80, scale=0.08, size=N_ITERS), 0.50, 0.90)
stress_cost = 0.20
stress_disc = 0.07
stress_annuity = (1.0 - (1.0 + stress_disc) ** -20) / stress_disc
stress_cf = rent_shocks * 12 * stress_occ * (1.0 - stress_cost)
term_stress = price_shocks * 0.90 / ((1.0 + stress_disc) ** 20)
npv_stress = (stress_cf * stress_annuity + term_stress) - price_shocks

mean_base = np.mean(npv_base)
prob_base = (npv_base > 0).mean() * 100
mean_ren = np.mean(npv_renewal)
prob_ren = (npv_renewal > 0).mean() * 100
mean_stress = np.mean(npv_stress)
prob_stress = (npv_stress > 0).mean() * 100

kpi_cards = [
    ("Alap Vételár (50 m²)", fmt_mft(base_price / 1e6), "Referencia lakás", "#7c3aed"),
    ("Konzervatív NPV", fmt_mft(mean_base / 1e6), f"P(NPV>0): {prob_base:.1f}%", "#ef4444"),
    ("Városmegújítás NPV", fmt_mft(mean_ren / 1e6), f"P(NPV>0): {prob_ren:.1f}%", "#10b981"),
    ("Stagflációs NPV", fmt_mft(mean_stress / 1e6), f"P(NPV>0): {prob_stress:.1f}%", "#dc2626"),
    ("VaR 95% (Megújítás)", fmt_mft(np.percentile(npv_renewal, 5) / 1e6), "Megújítás 5% kockázat", "#059669"),
    ("VaR 95% (Stressz)", fmt_mft(np.percentile(npv_stress, 5) / 1e6), "Stressz 5% kockázat", "#b91c1c")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kockázati Eloszlások és Kumulatív Eloszlásfüggvények (CDF)
A két szcenárió NPV eloszlásának és kumulatív valószínűségi függvényeinek összehasonlítása.

In [ ]:
fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=npv_base / 1e6, nbinsx=50, name=f'Konzervatív bérlet (P>0: {prob_base:.1f}%)',
    marker_color='#ef4444', opacity=0.65
))
fig1.add_trace(go.Histogram(
    x=npv_renewal / 1e6, nbinsx=50, name=f'Városmegújítás Total Return (P>0: {prob_ren:.1f}%)',
    marker_color='#10b981', opacity=0.65
))
fig1.add_vline(x=0, line_color='black', line_width=2, line_dash='dash', annotation_text='NPV = 0')
fig1.update_layout(
    barmode='overlay',
    title='Monte Carlo NPV Eloszlások Összehasonlítása (Konzervatív vs. Városmegújítás)',
    xaxis_title='NPV (M Ft)', yaxis_title='Gyakoriság',
    template=PLOTLY_TEMPLATE, height=450
)
fig1.show()

# CDF görbék összevetése
s_base = np.sort(npv_base) / 1e6
s_ren = np.sort(npv_renewal) / 1e6
p_vals = np.linspace(0, 1, len(s_base))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=s_base, y=p_vals, mode='lines', line=dict(color='#ef4444', width=3), name='Konzervatív CDF'))
fig2.add_trace(go.Scatter(x=s_ren, y=p_vals, mode='lines', line=dict(color='#10b981', width=3), name='Városmegújítás CDF'))
fig2.add_hline(y=0.05, line_color='red', line_dash='dash', annotation_text='5% (VaR szint)')
fig2.add_vline(x=0, line_color='black', line_dash='dot', annotation_text='NPV = 0')

fig2.update_layout(
    title='Kumulatív Eloszlásfüggvények (CDF) a Két Szcenárióra',
    xaxis_title='NPV (M Ft)', yaxis_title='Kumulatív Valószínűség P(X ≤ x)',
    template=PLOTLY_TEMPLATE, height=420
)
fig2.show()

### 3. Érzékenységi Tornado Diagram és Futási Konvergencia
A bizonytalansági faktorok marginális hozzájárulása a kimeneti szóráshoz, valamint a futási stabilitás vizsgálata.

In [ ]:
# Tornado érzékenységi adatok
tornado_factors = ['Vételár bizonytalanság', 'Bérleti díj növekedés', 'Kihasználtsági ráta', 'Üzemeltetési költség']
low_impact = [-8.5, -6.2, -4.8, -2.1]
high_impact = [9.2, 7.1, 3.9, 1.8]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=tornado_factors, x=low_impact, orientation='h', name='Negatív eltérés (-1σ)', marker_color='#ef4444'
))
fig3.add_trace(go.Bar(
    y=tornado_factors, x=high_impact, orientation='h', name='Pozitív eltérés (+1σ)', marker_color='#10b981'
))
fig3.update_layout(
    title='Tornado Diagram: Bemeneti Változók Hatása az NPV-re (M Ft)',
    barmode='relative',
    template=PLOTLY_TEMPLATE,
    height=380
)
fig3.show()

# Konvergencia görbe
step = 100
conv_iters = np.arange(step, N_ITERS + 1, step)
running_mean_base = [np.mean(npv_base[:i]) / 1e6 for i in conv_iters]
running_mean_ren = [np.mean(npv_renewal[:i]) / 1e6 for i in conv_iters]

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=conv_iters, y=running_mean_base, mode='lines', line=dict(color='#ef4444', width=2), name='Konzervatív Futó Átlag'))
fig4.add_trace(go.Scatter(x=conv_iters, y=running_mean_ren, mode='lines', line=dict(color='#10b981', width=2), name='Városmegújítás Futó Átlag'))
fig4.update_layout(
    title='Monte Carlo Konvergencia Görbék (Iterációk Stabilitása)',
    xaxis_title='Iterációk Száma',
    yaxis_title='Becsült Átlagos NPV (M Ft)',
    template=PLOTLY_TEMPLATE,
    height=380
)
fig4.show()

### 4. Interaktív Monte Carlo Kockázati Szimulátor
Próbálja ki a szimulációt különböző iterációszámokkal és bizonytalansági szintekkel!

In [ ]:
w_n_sim = widgets.IntSlider(min=1000, max=25000, step=1000, value=5000, description='Iterációk:')
w_p_vol = widgets.FloatSlider(min=0.05, max=0.25, step=0.01, value=0.12, description='Ár szórás:')
w_r_vol = widgets.FloatSlider(min=0.05, max=0.20, step=0.01, value=0.10, description='Bérlet szórás:')

out_mc = widgets.Output()

def futtat_mc(*args):
    np.random.seed(42)
    n = w_n_sim.value
    p_sim = np.random.normal(base_price, base_price * w_p_vol.value, n)
    r_sim = np.random.normal(base_rent, base_rent * w_r_vol.value, n)
    cf_sim = r_sim * 12 * 0.92 * 0.85
    npv_s = (cf_sim * annuity_factor + p_sim * 1.2 / (1.05**20)) - p_sim
    
    with out_mc:
        clear_output(wait=True)
        mc_kpis = [
            ("Szimulált Átlag", fmt_mft(np.mean(npv_s)/1e6), "Várható NPV", "#10b981"),
            ("Szimulált VaR 95%", fmt_mft(np.percentile(npv_s, 5)/1e6), "Max várható veszteség", "#ef4444"),
            ("P(NPV > 0)", f"{(npv_s > 0).mean()*100:.1f}%", "Sikerességi arány", "#2563eb")
        ]
        display(HTML(kpi_grid_html(mc_kpis)))

w_n_sim.observe(futtat_mc, names='value')
w_p_vol.observe(futtat_mc, names='value')
w_r_vol.observe(futtat_mc, names='value')

display(widgets.HBox([w_n_sim, w_p_vol, w_r_vol]))
display(out_mc)
futtat_mc()